# Personal Finance & Spending Analysis
## 02 — Data Cleaning

**Objective:** Prepare consistent data for analysis while preserving the original source.

**Input:** `data/raw_data.csv`  
**Outputs:** `data/cleaned_data.csv`, `reports/data_cleaning_summary.csv`

Principles: never overwrite raw data; remove exact duplicates; standardize names and categories; safely convert numeric fields; flag suspicious values; retain missing values unless a documented rule supports imputation.

## 1. Import Libraries and Configure Paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
from pathlib import Path
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "data").exists() else HERE.parent
print("Project root:", PROJECT_ROOT.resolve())

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "data_cleaning_summary.csv"
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Raw input:", RAW_DATA_PATH.resolve())
print("Clean output:", CLEAN_DATA_PATH.resolve())

## 2. Load Raw Dataset

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {RAW_DATA_PATH.resolve()}")
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]:,} columns.")
display(df_raw.head())

## 3. Create a Working Copy

In [ ]:
df = df_raw.copy(deep=True)
original_columns = df.columns.tolist()
print("Working copy created. Raw data remains unchanged.")

## 4. Standardize Column Names

In [ ]:
new_columns = (
    df.columns.astype(str).str.strip().str.lower()
      .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
      .str.strip("_")
)
duplicate_names = new_columns[new_columns.duplicated()].tolist()
if duplicate_names:
    raise ValueError(f"Duplicate names after standardization: {duplicate_names}")
df.columns = new_columns
column_audit = pd.DataFrame({
    "original_column": original_columns,
    "standardized_column": df.columns
})
display(column_audit)

## 5. Remove Exact Duplicate Rows

In [ ]:
rows_before = len(df)
duplicate_rows_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Rows before: {rows_before:,}")
print(f"Exact duplicates removed: {duplicate_rows_removed:,}")
print(f"Rows after: {len(df):,}")

## 6. Clean Text and Categorical Values

In [ ]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace("", pd.NA)

category_log = []
for column in ["occupation", "city_tier"]:
    if column in df.columns:
        before_n = int(df[column].nunique(dropna=True))
        df[column] = df[column].str.replace(r"\\s+", "_", regex=True).str.lower()
        after_n = int(df[column].nunique(dropna=True))
        category_log.append({
            "column": column,
            "unique_values_before": before_n,
            "unique_values_after": after_n
        })
        display(df[column].value_counts(dropna=False).to_frame("record_count"))
display(pd.DataFrame(category_log))

## 7. Convert Expected Numeric Columns

In [ ]:
expected_numeric_columns = [
    "income", "age", "dependents", "rent", "loan_repayment", "insurance",
    "groceries", "transport", "eating_out", "entertainment", "utilities",
    "healthcare", "education", "miscellaneous", "desired_savings_percentage",
    "desired_savings", "disposable_income", "potential_savings_groceries",
    "potential_savings_transport", "potential_savings_eating_out",
    "potential_savings_entertainment", "potential_savings_utilities",
    "potential_savings_healthcare", "potential_savings_education",
    "potential_savings_miscellaneous"
]
conversion_log = []
for column in expected_numeric_columns:
    if column in df.columns:
        before_non_missing = df[column].notna()
        converted = pd.to_numeric(df[column], errors="coerce")
        coerced = int((before_non_missing & converted.isna()).sum())
        infinite_mask = converted.notna() & ~np.isfinite(converted)
        infinite_count = int(infinite_mask.sum())
        converted.loc[infinite_mask] = np.nan
        df[column] = converted
        conversion_log.append({
            "column": column,
            "values_coerced_to_missing": coerced,
            "infinite_values_set_to_missing": infinite_count
        })
conversion_log_df = pd.DataFrame(
    conversion_log,
    columns=["column", "values_coerced_to_missing", "infinite_values_set_to_missing"]
)
display(conversion_log_df)

## 8. Identify Invalid and Suspicious Values
Negative financial amounts are set to missing under this project's assumption. If the source uses negatives for refunds or credits, revise this rule. Age and dependents are flagged, not automatically removed.

In [ ]:
non_negative_columns = [
    "income", "rent", "loan_repayment", "insurance", "groceries", "transport",
    "eating_out", "entertainment", "utilities", "healthcare", "education",
    "miscellaneous", "desired_savings", "disposable_income",
    "potential_savings_groceries", "potential_savings_transport",
    "potential_savings_eating_out", "potential_savings_entertainment",
    "potential_savings_utilities", "potential_savings_healthcare",
    "potential_savings_education", "potential_savings_miscellaneous"
]
negative_log = []
for column in non_negative_columns:
    if column in df.columns:
        mask = df[column] < 0
        negative_log.append({
            "column": column,
            "negative_values_found": int(mask.sum())
        })
        df.loc[mask, column] = np.nan
negative_log_df = pd.DataFrame(
    negative_log, columns=["column", "negative_values_found"]
)
display(negative_log_df)

In [ ]:
invalid_savings_percentage_count = 0
if "desired_savings_percentage" in df.columns:
    invalid_pct = (
        (df["desired_savings_percentage"] < 0) |
        (df["desired_savings_percentage"] > 100)
    )
    invalid_savings_percentage_count = int(invalid_pct.sum())
    df.loc[invalid_pct, "desired_savings_percentage"] = np.nan

if "age" in df.columns:
    df["age_review_flag"] = df["age"].notna() & ((df["age"] < 0) | (df["age"] > 100))
if "dependents" in df.columns:
    df["dependents_review_flag"] = df["dependents"].notna() & (
        (df["dependents"] < 0) | (df["dependents"] % 1 != 0)
    )

print("Invalid savings percentages set to missing:", invalid_savings_percentage_count)
for flag in ["age_review_flag", "dependents_review_flag"]:
    if flag in df.columns:
        print(f"{flag}: {int(df[flag].sum())} records")

## 9. Review Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
}).sort_values("missing_percentage", ascending=False)
display(missing_summary)
print("Missing values retained; no blanket imputation applied.")

## 10. Validate the Cleaned Dataset

In [ ]:
remaining_negative_count = int(sum(
    (df[c] < 0).sum() for c in non_negative_columns if c in df.columns
))
validation = {
    "rows_after_cleaning": len(df),
    "columns_after_cleaning": df.shape[1],
    "remaining_exact_duplicates": int(df.duplicated().sum()),
    "missing_cells": int(df.isna().sum().sum()),
    "negative_values_remaining_in_financial_fields": remaining_negative_count
}
display(pd.DataFrame(list(validation.items()), columns=["check", "result"]))

In [ ]:
expected_core_columns = [
    "income", "age", "dependents", "occupation", "city_tier",
    "rent", "groceries", "transport", "desired_savings", "disposable_income"
]
core_check = pd.DataFrame({
    "column": expected_core_columns,
    "present": [c in df.columns for c in expected_core_columns]
})
display(core_check)
missing_core = core_check.loc[~core_check["present"], "column"].tolist()
if missing_core:
    print("Review missing expected columns:", missing_core)
else:
    print("All expected core columns are present.")

## 11. Compare Before and After Cleaning

In [ ]:
comparison = pd.DataFrame({
    "metric": ["Rows", "Columns", "Missing cells", "Exact duplicate rows"],
    "before_cleaning": [
        len(df_raw), df_raw.shape[1],
        int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum())
    ],
    "after_cleaning": [
        len(df), df.shape[1],
        int(df.isna().sum().sum()), int(df.duplicated().sum())
    ]
})
display(comparison)

## 12. Save Cleaned Dataset and Audit Report

In [ ]:
report_items = [
    ("raw_row_count", len(df_raw)),
    ("cleaned_row_count", len(df)),
    ("raw_column_count", df_raw.shape[1]),
    ("cleaned_column_count", df.shape[1]),
    ("exact_duplicate_rows_removed", duplicate_rows_removed),
    ("missing_cells_before", int(df_raw.isna().sum().sum())),
    ("missing_cells_after", int(df.isna().sum().sum())),
    ("remaining_exact_duplicates", int(df.duplicated().sum())),
    ("invalid_savings_percentages_set_to_missing", invalid_savings_percentage_count),
    ("negative_values_remaining_in_financial_fields", remaining_negative_count)
]
for item in negative_log:
    report_items.append((f"negative_values_found_{item['column']}", item["negative_values_found"]))
for item in conversion_log:
    report_items.append((f"coerced_to_missing_{item['column']}", item["values_coerced_to_missing"]))
    report_items.append((f"infinite_to_missing_{item['column']}", item["infinite_values_set_to_missing"]))
for flag in ["age_review_flag", "dependents_review_flag"]:
    if flag in df.columns:
        report_items.append((f"{flag}_count", int(df[flag].sum())))

cleaning_report = pd.DataFrame(report_items, columns=["check", "result"])
df.to_csv(CLEAN_DATA_PATH, index=False)  # New file; raw_data.csv is never overwritten.
cleaning_report.to_csv(REPORT_PATH, index=False)
print("Cleaned data saved:", CLEAN_DATA_PATH.resolve())
print("Cleaning report saved:", REPORT_PATH.resolve())
display(cleaning_report)

## 13. Final Review

- Confirm the raw CSV remains unchanged.
- Review age and dependents flags.
- Review numeric conversion and invalid-value counts.
- Check that expected columns are present.
- Do not impute missing values without a documented business rule.

**Business-rule caveat:** The source formulas for `disposable_income`, `desired_savings`, `desired_savings_percentage`, and `potential_savings_*` have not been independently verified. This notebook preserves these source fields.

**Next:** Run `03_eda_analysis.ipynb`.